# 🔍 Interactive Solar Panel Model Testing & Comparison
### Test **YOLO11m-seg**, **Mask R-CNN**, and **CNN U-Net** on Any Image

Upload any test satellite image (or select one from the dataset) to run inference on all 3 trained models and compare their predictions side-by-side:

| Model | Type | Output |
|:---|:---|:---|
| **YOLO11m-seg** | Instance Segmentation | Individual panel polygons + confidence scores |
| **Mask R-CNN** | Two-stage Instance Seg | Bounding boxes + instance masks |
| **CNN U-Net** | Semantic Pixel Seg | Dense continuous solar probability heatmap |

--- 
## 1. Install Dependencies & Setup Environment

In [ ]:
# Verify GPU
!nvidia-smi

# 1. Install YOLO and CNN dependencies
!pip install -q ultralytics segmentation-models-pytorch albumentations opencv-python matplotlib

# 2. Install Detectron2 (Mask R-CNN)
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference Device: {device}")

--- 
## 2. Load Weights for All 3 Models
Upload your weights (`yolo11m-seg.pt`, `mask_rcnn_final.pth`, `best_cnn_unet.pth`) or clone them if uploaded to Drive/GitHub.

In [ ]:
# ==========================================
# 1. Load YOLO11m-seg
# ==========================================
from ultralytics import YOLO
for _p in ["yolo11m-seg.pt", "yolo11m_best.pt", "Training results/weights/yolo11m_best.pt", "Training results/yolo11m-seg.pt"]:
    if os.path.exists(_p):
        yolo_weights = _p
        break
else:
    yolo_weights = "yolo11m-seg.pt"
if os.path.exists(yolo_weights):
    yolo_model = YOLO(yolo_weights)
    print("✅ YOLO11m-seg loaded successfully.")
else:
    print(f"⚠️ Please upload '{yolo_weights}' in the Colab file explorer.")

# ==========================================
# 2. Load Mask R-CNN (Detectron2)
# ==========================================
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

for _p in ["mask_rcnn_final.pth", "Training results/weights/mask_rcnn_final.pth", "Training results/mask_rcnn_final.pth"]:
    if os.path.exists(_p):
        mask_rcnn_weights = _p
        break
else:
    mask_rcnn_weights = "mask_rcnn_final.pth"
mask_rcnn_predictor = None
if os.path.exists(mask_rcnn_weights):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
    cfg.MODEL.WEIGHTS = mask_rcnn_weights
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    mask_rcnn_predictor = DefaultPredictor(cfg)
    print("✅ Mask R-CNN loaded successfully.")
else:
    print(f"⚠️ Please upload '{mask_rcnn_weights}' in the Colab file explorer.")

# ==========================================
# 3. Load CNN U-Net (ResNet-34)
# ==========================================
import segmentation_models_pytorch as smp
for _p in ["best_cnn_unet.pth", "Training results/weights/best_cnn_unet.pth", "Training results/best_cnn_unet.pth"]:
    if os.path.exists(_p):
        cnn_weights = _p
        break
else:
    cnn_weights = "best_cnn_unet.pth"
cnn_model = None
if os.path.exists(cnn_weights):
    cnn_model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )
    cnn_model.load_state_dict(torch.load(cnn_weights, map_location=device))
    cnn_model.to(device)
    cnn_model.eval()
    print("✅ CNN U-Net loaded successfully.")
else:
    print(f"⚠️ Please upload '{cnn_weights}' in the Colab file explorer.")

--- 
## 3. Upload a Custom Image to Test
Run this cell to select an image from your computer.

In [ ]:
from google.colab import files

print("Upload one or more satellite images (.jpg / .png):")
uploaded = files.upload()

test_image_path = list(uploaded.keys())[0] if uploaded else None
if test_image_path:
    print(f"Selected image for testing: {test_image_path}")

*(Alternatively, if you cloned the `FYP` dataset, you can test directly on any dataset image below):*

In [ ]:
# Optional: Pick a sample from the test set (clones from GitHub if needed)
import os, glob

if not os.path.exists("FYP") and not os.path.exists("Dataset"): 
    print("Cloning dataset from GitHub repository (Hinarashid220/FYP)...")
    !git clone https://github.com/Hinarashid220/FYP.git

dataset_samples = (
    glob.glob("FYP/Dataset/YOLO/2025/test/images/*.jpg") or
    glob.glob("Dataset/YOLO/2025/test/images/*.jpg") or
    glob.glob("FYP/Dataset/COCO/2025/test/images/*.jpg") or
    glob.glob("Dataset/COCO/2025/test/images/*.jpg")
)
if dataset_samples:
    test_image_path = dataset_samples[0] # Pick the first test image
    print(f"Using test set image: {test_image_path}")

--- 
## 4. Run Simultaneous Inference & Side-by-Side Comparison

In [ ]:
assert test_image_path and os.path.exists(test_image_path), "Please upload or select an image in Step 3!"

# Read image
orig_img_bgr = cv2.imread(test_image_path)
orig_img_rgb = cv2.cvtColor(orig_img_bgr, cv2.COLOR_BGR2RGB)
h, w = orig_img_rgb.shape[:2]

# -----------------------------------------------------
# 1. YOLO11m-seg Inference
# -----------------------------------------------------
yolo_vis = orig_img_rgb.copy()
yolo_count = 0
if 'yolo_model' in globals():
    yolo_res = yolo_model.predict(orig_img_rgb, conf=0.25, imgsz=432, verbose=False)[0]
    if yolo_res.masks is not None:
        yolo_count = len(yolo_res.masks.data)
        # Overlay polygon masks on image
        yolo_vis = yolo_res.plot(boxes=True, labels=True, probs=False)

# -----------------------------------------------------
# 2. Mask R-CNN Inference
# -----------------------------------------------------
mask_rcnn_vis = orig_img_rgb.copy()
mask_rcnn_count = 0
if mask_rcnn_predictor is not None:
    from detectron2.utils.visualizer import Visualizer, ColorMode
    outputs = mask_rcnn_predictor(orig_img_bgr)
    instances = outputs["instances"].to("cpu")
    mask_rcnn_count = len(instances)
    v = Visualizer(orig_img_rgb, scale=1.0, instance_mode=ColorMode.IMAGE)
    out = v.draw_instance_predictions(instances)
    mask_rcnn_vis = out.get_image()

# -----------------------------------------------------
# 3. CNN U-Net Inference
# -----------------------------------------------------
cnn_heatmap = np.zeros((h, w), dtype=np.float32)
cnn_vis = orig_img_rgb.copy()
if cnn_model is not None:
    # Preprocess
    resized = cv2.resize(orig_img_rgb, (448, 448))
    norm_img = (resized / 255.0 - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
    tensor_in = torch.tensor(norm_img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = cnn_model(tensor_in)
        prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    cnn_heatmap = cv2.resize(prob, (w, h))
    # Create binary mask overlay
    bin_mask = (cnn_heatmap > 0.5).astype(np.uint8)
    overlay = orig_img_rgb.copy()
    overlay[bin_mask == 1] = [255, 50, 50]  # Highlight detected solar in bright red
    cnn_vis = cv2.addWeighted(orig_img_rgb, 0.6, overlay, 0.4, 0)

# -----------------------------------------------------
# Side-by-Side Plotting
# -----------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(orig_img_rgb)
axes[0].set_title(f"Original Satellite Input\n({Path(test_image_path).name})", fontsize=13, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(yolo_vis)
axes[1].set_title(f"YOLO11m-seg\n({yolo_count} solar arrays detected)", fontsize=13, fontweight="bold", color="navy")
axes[1].axis("off")

axes[2].imshow(mask_rcnn_vis)
axes[2].set_title(f"Mask R-CNN (ResNet-50)\n({mask_rcnn_count} instances detected)", fontsize=13, fontweight="bold", color="darkgreen")
axes[2].axis("off")

axes[3].imshow(cnn_vis)
axes[3].set_title(f"CNN U-Net (ResNet-34)\n(Dense Solar Heatmap)", fontsize=13, fontweight="bold", color="darkred")
axes[3].axis("off")

plt.tight_layout()
plt.savefig("model_manual_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("=" * 70)
print(f"TEST SUMMARY FOR: {Path(test_image_path).name}")
print(f"  • YOLO11m-seg Detected: {yolo_count} installations")
print(f"  • Mask R-CNN Detected : {mask_rcnn_count} installations")
print(f"  • CNN Estimated Area  : {np.sum(cnn_heatmap > 0.5):,} pixels of solar panels")
print("=" * 70)